In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)


import gc
import ast
import torch
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset
from peft import PeftModel
from peft import LoraConfig
from peft import get_peft_model
from transformers import Trainer
from transformers import TrainingArguments
from transformers import Blip2Processor, Blip2ForConditionalGeneration

In [ ]:
train_img_root = './blip/data/train'
test_img_root = './blip/data/test'

# 1. 基本加载
df = pd.read_csv('./blip/data/processed_dataset.csv')
df['caption'] = df['raw'].apply(ast.literal_eval)  # 将字符串转换为列表
# 2. 根据 'split' 列进行划分
train_df = df[df['split'] == 'train']
# print(train_df.head())  # 查看前5行
test_df = df[df['split'] == 'test']
# print(test_df.head())  # 查看前5行

train_data = []

for i in range(len(train_df)):
    img_path = train_df.iloc[i]['filename']
    caption = train_df.iloc[i]['caption']
    for cap in caption[0:1]:
        train_data.append({
            "image": os.path.join(train_img_root, img_path),
            "text": cap.lower()
        })
print(train_data)
print("Length of train_data:", len(train_data))

test_data = []

for i in range(len(test_df)):
    img_path = test_df.iloc[i]['filename']
    caption = test_df.iloc[i]['caption']
    for cap in caption[0:1]:
        test_data.append({
            "image": os.path.join(test_img_root, img_path),
            "text": cap.lower()
        })
print(test_data) 
print("Length of test_data:", len(test_data))

In [ ]:
model_name = "Salesforce/blip2-flan-t5-xl"
cache_dir = "./model/blip/blip2_cache"
dtype = torch.bfloat16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# processor
processor = Blip2Processor.from_pretrained(
    model_name,
    cache_dir=cache_dir
    )
# model
model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=dtype,
    ).to(device)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q", "k", "v", "o",       # T5 attention
        "query", "key", "value"   # Q-Former
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"模型设备检查: {next(model.parameters()).device}")


In [ ]:
class FlickrDataset(Dataset):
    def __init__(self, data, processor, max_length=128):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # 1. 加载图片
        try:
            image = Image.open(item["image"]).convert("RGB")
        except Exception as e:
            print(f"Error loading image {item['image']}: {e}")
            image = Image.new('RGB', (224, 224))

        # 2. 准备文本
        prompt = "Question: Describe this image. Answer:"
        caption = item["text"]
        full_text = prompt + " " + caption

        # 3. 处理图片
        image_inputs = self.processor(images=image, return_tensors="pt")

        # 4. 处理文本 (手动构建 Labels 以确保正确)
        # 第一步：对完整文本进行 tokenize
        text_inputs = self.processor.tokenizer(
            text=full_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        # 第二步：构建 Labels
        # 我们需要将 Prompt 部分设为 -100，只计算 Caption 的 Loss
        # 先复制一份 input_ids 作为 labels
        labels = text_inputs["input_ids"].clone()

        # 第三步：找到 Prompt 的结束位置
        # 对 Prompt 单独 tokenize，看它有多长
        prompt_tokens = self.processor.tokenizer(
            text=prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=self.max_length
        )
        prompt_length = prompt_tokens["input_ids"].shape[1]

        # 第四步：Mask 掉 Prompt 部分 (设为 -100)
        # 只有 prompt_length 之后的 token 才参与 Loss 计算
        labels[:, :prompt_length] = -100

        # 第五步：Mask 掉 Padding 部分 (设为 -100)
        # 通常 tokenizer 会用 pad_token_id (通常是 0)
        # 如果 label 等于 pad_token_id，也设为 -100 (可选，视具体模型而定，推荐开启)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        # 5. 整理输出
        inputs = {}
        inputs["pixel_values"] = image_inputs["pixel_values"]
        inputs["input_ids"] = text_inputs["input_ids"]
        inputs["attention_mask"] = text_inputs["attention_mask"]
        inputs["labels"] = labels

        # 去除 batch 维度
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        
        return inputs

# ==========================================
# 验证修正后的效果
# ==========================================
# 实例化
train_dataset = FlickrDataset(train_data, processor)
sample = train_dataset[0]

print("Input IDs shape:", sample['input_ids'].shape)
print("Labels shape:", sample['labels'].shape)

# 检查 Labels 中有多少有效的 token (不是 -100)
valid_labels = sample['labels'][sample['labels'] != -100]
print(f"有效 Label 数量: {len(valid_labels)}")

# 解码查看
print("Decoded Input:", processor.decode(sample['input_ids'], skip_special_tokens=True))
print("Decoded Labels (Only Caption part):", processor.decode(valid_labels, skip_special_tokens=True))

training_args = TrainingArguments(
    output_dir="./blip/blip2_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    num_train_epochs=3,
    fp16=False,
    bf16=True,
    logging_steps=20,
    save_steps=500,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

In [ ]:
model.save_pretrained("./blip/blip2_lora")
processor.save_pretrained("./blip/blip2_lora")